# Chuẩn bị dữ liệu cho bài toán dự đoán mua hàng

Sau bước tìm hiểu dữ liệu, notebook này tạo bảng đầu vào cho mô hình. Tại mốc chốt `s`, ta chỉ nhìn lịch sử đến ngày `s`. Nhãn bằng 1 nếu hộ có **bất kỳ giỏ hàng nào** trong 7 ngày sau đó, kể cả giỏ nhiên liệu, nhóm hàng khác hoặc giỏ có doanh số bằng 0.

Mỗi dòng đầu ra là một hộ tại một mốc chốt. Các mốc cách nhau 7 ngày, từ ngày 180 đến 698; chỉ xét hộ đã có giao dịch đến mốc đó. Cửa sổ nhìn lại dài tối đa 180 ngày, **không có nghĩa** mỗi hộ đều đã xuất hiện đủ 180 ngày.

Phần này chỉ chuẩn bị và kiểm tra dữ liệu, chưa huấn luyện mô hình. Tập kiểm tra cuối đã từng được xem trong quá trình làm dự án nên không thể coi là hoàn toàn độc lập.

## 1. Đọc giao dịch và kiểm tra mã giỏ hàng

Trước tiên chỉ lấy các cột giao dịch cần thiết. Chưa cộng `QUANTITY` vì đơn vị đo không rõ ở mọi nhóm hàng. Chưa dùng thông tin hộ và chiến dịch vì không biết chắc chúng đã có ở từng mốc chốt trong quá khứ. Giữ mọi giỏ hàng, không lọc theo doanh số hay nhóm hàng.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'sample_data' / 'dunnhumby' / 'transaction_data.csv'
OUT = ROOT / 'artifacts' / 'data_preparation'
tx = pd.read_csv(DATA, usecols=['household_key', 'BASKET_ID', 'DAY', 'SALES_VALUE'])
assert not tx[['household_key', 'BASKET_ID', 'DAY', 'SALES_VALUE']].isna().any().any()
assert tx.DAY.min() >= 1
basket_scope = tx.groupby('BASKET_ID').agg(households=('household_key','nunique'), days=('DAY','nunique'))
assert basket_scope.households.eq(1).all() and basket_scope.days.eq(1).all()
baskets = (tx.groupby(['household_key','BASKET_ID','DAY'], as_index=False, sort=False)
           .agg(sales=('SALES_VALUE','sum'), line_items=('SALES_VALUE','size')))
assert not baskets.duplicated(['household_key','BASKET_ID','DAY']).any()
print(f'{len(tx):,} dòng giao dịch → {len(baskets):,} basket → {baskets.household_key.nunique():,} household')
print('DAY:', tx.DAY.min(), 'đến', tx.DAY.max())

2,595,732 dòng giao dịch → 276,484 basket → 2,500 household
DAY: 1 đến 711


## 2. Từ giỏ hàng đến lịch sử mua theo ngày của từng hộ

Một ngày được tính là có mua nếu hộ có ít nhất một giỏ. Dù mua nhiều giỏ trong cùng ngày, số ngày có mua vẫn chỉ tăng một. `sales` là tiền nhà bán lẻ nhận được theo dữ liệu, có thể khác số tiền khách thực trả.

In [2]:
daily = (baskets.groupby(['household_key','DAY'], as_index=False)
         .agg(baskets=('BASKET_ID','size'), sales=('sales','sum'), line_items=('line_items','sum')))
daily['active_days'] = 1
assert not daily.duplicated(['household_key','DAY']).any()
assert daily.baskets.sum() == len(baskets)
display(daily.head())

,household_key,DAY,baskets,sales,line_items,active_days
0,1,51,1,78.66,30,1
1,1,67,1,41.10,12,1
2,1,88,1,26.90,12,1
3,1,94,1,63.43,23,1
4,1,101,1,53.45,17,1


## 3. Tạo một dòng cho mỗi hộ tại mỗi mốc chốt

Với từng khoảng nhìn lại 7, 30, 90 và 180 ngày, tính số giỏ, số ngày có mua, số dòng sản phẩm và tổng `SALES_VALUE` tính đến mốc `s`. `recency_days` là số ngày từ lần mua gần nhất; `observed_tenure_days` là số ngày từ lần đầu thấy hộ trong dữ liệu. Mã hộ và ngày chốt không tự động trở thành biến dự báo.

Nhãn dự báo dùng 7 ngày **sau** mốc chốt: giỏ ở ngày `s` thuộc lịch sử; giỏ ở ngày `s+7` được tính vào nhãn; giỏ ở `s+8` thì không.

In [3]:
HORIZON = 7
SNAPSHOTS = list(range(180, 699, 7))
WINDOWS = (7, 30, 90, 180)
max_day = int(daily.DAY.max())
assert max(SNAPSHOTS) + HORIZON <= max_day
households = np.sort(daily.household_key.unique())
matrices = {}
for metric in ['baskets','sales','line_items','active_days']:
    matrices[metric] = (daily.pivot(index='household_key', columns='DAY', values=metric)
                        .reindex(index=households, columns=range(max_day+1))
                        .fillna(0).to_numpy(dtype=np.float64))
cumulative = {metric: np.cumsum(matrix, axis=1) for metric, matrix in matrices.items()}
activity = matrices['baskets'] > 0
day_axis = np.arange(max_day+1)[None, :]
first = np.where(activity, day_axis, max_day+1).min(axis=1)
last = np.maximum.accumulate(np.where(activity, day_axis, -1), axis=1)
frames = []
for s in SNAPSHOTS:
    values = {'household_key': households, 'snapshot_day': s,
              'recency_days': s-last[:,s], 'observed_tenure_days': s-first}
    for w in WINDOWS:
        for metric in matrices:
            values[f'{metric}_{w}d'] = cumulative[metric][:,s] - cumulative[metric][:,s-w]
    values['target_any_basket'] = ((cumulative['baskets'][:,s+HORIZON] - cumulative['baskets'][:,s]) > 0).astype('int8')
    frames.append(pd.DataFrame(values).loc[first <= s])
panel = pd.concat(frames, ignore_index=True)
assert not panel.duplicated(['household_key','snapshot_day']).any()
assert panel.recency_days.ge(0).all() and panel.observed_tenure_days.ge(0).all()
assert panel.target_any_basket.isin([0,1]).all()
for w in WINDOWS:
    assert panel[f'active_days_{w}d'].between(0,w).all()
    assert (panel[f'active_days_{w}d'] <= panel[f'baskets_{w}d']).all()
print('Tổng household–snapshot:', f'{len(panel):,}', '| Số cutoff:', len(SNAPSHOTS))

Tổng household–snapshot: 187,305 | Số cutoff: 75


## 4. Chia dữ liệu theo thứ tự thời gian

Giữ cách chia đã dùng trong dự án. Hai mốc 544 và 628 được bỏ trống để 7 ngày cần dự báo của tập trước kết thúc trước mốc đầu tiên của tập sau. Lịch sử của cùng một hộ ở các mốc khác nhau vẫn có thể trùng nhau; các dòng này không phải những hộ độc lập.

| Tập dữ liệu | Mốc chốt | Ngày cuối có thể dùng làm nhãn |
|---|---|---|
| Huấn luyện | 180–537 | đến 544 |
| Chọn và kiểm tra mô hình | 551–621 | đến 628 |
| Kiểm tra cuối | 635–698 | đến 705 |

Không dùng tập kiểm tra cuối để chọn biến hay chọn mô hình. Do tập này đã từng được xem trước đây, kết quả của nó chỉ mang tính tham khảo, không phải phép kiểm tra hoàn toàn mới.

In [4]:
def assign_split(s):
    if 180 <= s <= 537: return 'train'
    if 551 <= s <= 621: return 'validation'
    if 635 <= s <= 698: return 'test'
    return 'embargo'
panel['split'] = panel.snapshot_day.map(assign_split)
assert set(panel.loc[panel.split.eq('embargo'), 'snapshot_day']) == {544, 628}
assert panel.loc[panel.split.eq('train'), 'snapshot_day'].max()+HORIZON < panel.loc[panel.split.eq('validation'), 'snapshot_day'].min()
assert panel.loc[panel.split.eq('validation'), 'snapshot_day'].max()+HORIZON < panel.loc[panel.split.eq('test'), 'snapshot_day'].min()
summary = panel.groupby('split').agg(rows=('household_key','size'), households=('household_key','nunique'),
                                      cutoff_min=('snapshot_day','min'), cutoff_max=('snapshot_day','max'),
                                      cutoffs=('snapshot_day','nunique'))
display(summary.loc[['train','validation','test','embargo']])

,rows,households,cutoff_min,cutoff_max,cutoffs
split,,,,,
train,129834,2498,180,537,52
validation,27480,2499,551,621,11
test,24994,2500,635,698,10
embargo,4997,2499,544,628,2


## 5. Kiểm tra lại số liệu và ranh giới thời gian

Đối chiếu 300 dòng huấn luyện với giỏ hàng gốc: kiểm tra cả lịch sử và nhãn 7 ngày. Dùng thêm ví dụ đơn giản để kiểm tra ngày ở sát ranh giới. Vẫn giữ ngày 278 và 643 có rất ít giỏ; đánh dấu những mốc có nhãn liên quan để kiểm tra thêm, không mặc định nhãn của chúng sai.

In [5]:
def count_in_window(days, start, end):
    return ((days > start) & (days <= end)).sum()
boundary_days = np.array([10, 11, 17, 18])
assert count_in_window(boundary_days, 10, 17) == 2  # 11, 17; không gồm 10 hoặc 18
sample = panel.loc[panel.split.eq('train')].sample(300, random_state=42)
basket_by_household = {h: g for h,g in baskets.groupby('household_key', sort=False)}
for row in sample.itertuples(index=False):
    source = basket_by_household[row.household_key]
    future = source[(source.DAY > row.snapshot_day) & (source.DAY <= row.snapshot_day+HORIZON)]
    assert row.target_any_basket == int(not future.empty)
    for w in WINDOWS:
        history = source[(source.DAY > row.snapshot_day-w) & (source.DAY <= row.snapshot_day)]
        assert getattr(row, f'baskets_{w}d') == len(history)
        assert getattr(row, f'active_days_{w}d') == history.DAY.nunique()
        assert getattr(row, f'line_items_{w}d') == history.line_items.sum()
        assert np.isclose(getattr(row, f'sales_{w}d'), history.sales.sum(), atol=1e-6)
assert np.isfinite(panel.select_dtypes(include='number').to_numpy()).all()
panel['label_window_touches_anomaly'] = panel.snapshot_day.map(
    lambda s: any(s < d <= s+HORIZON for d in (278,643)))
print('PASS: ranh giới thời gian, 300 đối chiếu train, giá trị hữu hạn và tính duy nhất.')
display(panel.loc[panel.label_window_touches_anomaly].groupby(['split','snapshot_day']).size().rename('household_snapshot').to_frame())

PASS: ranh giới thời gian, 300 đối chiếu train, giá trị hữu hạn và tính duy nhất.


,,household_snapshot
split,snapshot_day,
test,642,2499
train,271,2496


### 5.1. Hộ đã xuất hiện bao lâu và có gặp ngày bất thường không?

`observed_tenure_days` đếm từ giỏ đầu tiên được ghi nhận. Trên **tập huấn luyện**, ta xem tỷ lệ mua tiếp theo thời gian hộ đã xuất hiện. Không coi thời gian trước giỏ đầu tiên là chắc chắn không mua; xuất hiện đủ 180 ngày cũng không chứng minh dữ liệu nguồn đầy đủ.

Cờ kiểm tra cho biết khoảng lịch sử trước mốc chốt có chứa ngày 278 hoặc 643. Cờ chỉ nêu khả năng bị ảnh hưởng, không chứng minh dữ liệu bị thiếu hoặc cho biết sai lệch bao nhiêu.

In [6]:
panel['observed_days_inclusive'] = panel.observed_tenure_days + 1
train_audit = panel.loc[panel.split.eq('train')].copy()
train_audit['history_group'] = np.where(train_audit.observed_days_inclusive < 180,
                                        'Dưới 180 ngày', 'Từ 180 ngày')
history_summary = (train_audit.groupby('history_group')
                   .agg(household_snapshots=('household_key','size'),
                        unique_households=('household_key','nunique'),
                        target_rate=('target_any_basket','mean')))
display(history_summary)
for w in (90, 180):
    panel[f'history_{w}d_touches_anomaly'] = panel.snapshot_day.map(
        lambda s: any(s-w < d <= s for d in (278, 643)))
    counts = panel.loc[panel.split.eq('train')].groupby(f'history_{w}d_touches_anomaly').agg(
        household_snapshots=('household_key','size'), cutoffs=('snapshot_day','nunique'))
    print(f'Cửa sổ lịch sử {w} ngày — tập train:')
    display(counts)
panel.drop(columns=['observed_days_inclusive'], inplace=True)

,household_snapshots,unique_households,target_rate
history_group,,,
Dưới 180 ngày,24030,2483,0.519767
Từ 180 ngày,105804,2497,0.529791


Cửa sổ lịch sử 90 ngày — tập train:


,household_snapshots,cutoffs
history_90d_touches_anomaly,,
False,97373,39
True,32461,13


Cửa sổ lịch sử 180 ngày — tập train:


,household_snapshots,cutoffs
history_180d_touches_anomaly,,
False,64909,26
True,64925,26


Một hộ có thể xuất hiện ở nhiều nhóm thời điểm. Tỷ lệ mua tiếp khác nhau chưa chứng minh thời gian được quan sát là nguyên nhân; sau này nên báo cáo kết quả mô hình theo các nhóm này, cùng với kết quả toàn bộ tập.

## 6. Lưu các bảng đầu vào cho bước mô hình

Mỗi file chỉ chứa một tập thời gian; hai mốc bỏ trống không được đưa vào huấn luyện. Cờ `label_window_touches_anomaly`, mã hộ, ngày chốt, tên tập và nhãn là thông tin quản lý hoặc kết quả cần dự đoán, không tự động là biến đầu vào. Cờ này chỉ đánh dấu 7 ngày dùng làm nhãn; ngày bất thường cũng có thể nằm trong lịch sử.

Chưa tính giá trị để điền ô thiếu, chuẩn hóa hay chọn biến trên toàn bộ dữ liệu. Nếu cần những bước đó, chỉ học từ tập huấn luyện để tránh nhìn trước dữ liệu đánh giá.

In [7]:
OUT.mkdir(parents=True, exist_ok=True)
excluded = {'household_key','snapshot_day','split','target_any_basket','label_window_touches_anomaly','history_90d_touches_anomaly','history_180d_touches_anomaly'}
feature_columns = [c for c in panel.columns if c not in excluded]
for split in ['train','validation','test']:
    panel.loc[panel.split.eq(split)].to_parquet(OUT / f'{split}.parquet', index=False)
manifest = {'target': 'target_any_basket', 'horizon_days': HORIZON,
            'history_windows_days': list(WINDOWS), 'features': feature_columns,
            'metadata': sorted(excluded - {'target_any_basket'}),
            'embargo_cutoffs': [544,628],
            'split_boundaries': {'train':[180,537], 'validation':[551,621], 'test':[635,698]},
            'test_previously_inspected': True,
            'notes': ['Demographic and marketing excluded: historical availability unverified.',
                      'QUANTITY excluded; anomalies retained.',
                      'Anomaly flags mark label, 90-day and 180-day history windows.']}
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
for split in ['train','validation','test']:
    saved = pd.read_parquet(OUT / f'{split}.parquet')
    assert len(saved) == int(panel.split.eq(split).sum())
    assert saved.split.eq(split).all() and not saved.duplicated(['household_key','snapshot_day']).any()
print('Đã xuất:', OUT, '| Số biến đầu vào:', len(feature_columns))
# Hợp đồng dùng cho giai đoạn modeling: tách feature khỏi metadata.
import sys
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from propensity_v2.prepared_data import load_prepared_split
for split in ['train','validation','test']:
    checked, checked_features = load_prepared_split(OUT, split)
    assert checked_features == feature_columns
print('PASS: hợp đồng dữ liệu và danh sách biến đầu vào của từng tập.')

,household_snapshots,unique_households,target_rate
history_group,,,
Dưới 180 ngày,24030,2483,0.519767
Từ 180 ngày,105804,2497,0.529791


Cửa sổ lịch sử 90 ngày — tập train:


,household_snapshots,cutoffs
history_90d_touches_anomaly,,
False,97373,39
True,32461,13


Cửa sổ lịch sử 180 ngày — tập train:


,household_snapshots,cutoffs
history_180d_touches_anomaly,,
False,64909,26
True,64925,26


**Kết quả:** Ba bảng đã được kiểm tra ranh giới thời gian và đối chiếu với một mẫu giỏ hàng gốc. Báo cáo đánh giá cần tiếp tục nêu rõ hai ngày bất thường, thời điểm sẵn có của dữ liệu bổ sung và việc tập kiểm tra cuối đã từng được xem.

Bản xử lý V2 trước đây dùng bộ biến khác và có thông tin hộ; lệnh huấn luyện cũ đã được chặn mặc định để tránh dùng nhầm. Các file Parquet mới được kiểm tra bằng `propensity_v2.prepared_data.load_prepared_split`. Notebook này dừng ở bước chuẩn bị dữ liệu.